# Week 31, Day 4: Irregular firing: integrate-and-fire neurons with noisy, balanced input

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Softky and Koch's neuron

`lif(re, ri)` simulates 20 s and returns spike times (ms). Write `cv(spikes)`, then run the neuron with excitation only at r_e = 1591 Hz, which gives about 20 Hz, and report its rate `rate0` and CV `cv0`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lif(re, ri, we=0.5, wi=0.5, tau=20.0, Vth=15.0, T=20000.0, dt=0.1, seed=1, tref=2.0):
    r = np.random.default_rng(seed); n = int(T / dt)
    drive = we * r.poisson(re * dt / 1000, n) - wi * r.poisson(ri * dt / 1000, n)
    V, spikes, last, decay = 0.0, [], -1e9, np.exp(-dt / tau)
    for k in range(n):
        if k * dt - last < tref:
            continue
        V = V * decay + drive[k]
        if V >= Vth:
            spikes.append(k * dt); V = 0.0; last = k * dt
    return np.array(spikes)

def cv(spikes):
    return ...

s0 = lif(1591, 0)
rate0, cv0 = len(s0) / 20, cv(s0)
print(f"excitation only: {rate0:.1f} Hz, CV {cv0:.2f}")
plt.hist(np.diff(s0), 50); plt.xlabel("interspike interval (ms)"); plt.show()

In [ ]:
import numpy as np
def test_cv():
    "cv computes the standard deviation over the mean of the intervals"
    assert abs(cv(np.array([0.0, 10, 20, 30])) - 0) < 1e-12 and abs(cv(np.array([0.0, 10, 30])) - np.std([10, 20]) / 15) < 1e-12

def test_regular():
    "with excitation only, the neuron fires near 20 Hz with a CV below 0.45"
    assert 15 < rate0 < 25 and cv0 < 0.45

test_cv()
test_regular()
print("All checks passed")

### Exercise: Irregularity from balance

For the pairs (r_i, r_e) in `pairs`, simulate the neuron, and store its rate and CV in `results[r_i]`.

In [ ]:
pairs = [(0, 1591), (3000, 4373), (6000, 7247), (10000, 11107)]
results = {}
for ri, re in pairs:
    s = lif(re, ri)
    results[ri] = ...
    print(f"inhibition {ri:5d} Hz, excitation {re:5d} Hz: {results[ri][0]:.1f} Hz, CV {results[ri][1]:.2f}")

In [ ]:
import numpy as np
def test_rates():
    "all four conditions fire between 15 and 26 Hz"
    assert all(15 < results[ri][0] < 26 for ri in results)

def test_cv():
    "the CV rises with inhibition, from below 0.45 to above 0.6"
    cvs = [results[ri][1] for ri in sorted(results)]
    assert cvs[0] < 0.45 and cvs[-1] > 0.6 and all(a < b + 0.03 for a, b in zip(cvs, cvs[1:]))

test_rates()
test_cv()
print("All checks passed")

### Exercise: Mean-driven or fluctuation-driven?

For each pair, compute the free membrane potential's mean `mu[r_i]` and standard deviation `sigma[r_i]` (mV), with τ = 20 ms and rates converted from Hz to per ms. Check them against a simulation of the free voltage for the most balanced pair.

In [ ]:
mu, sigma = {}, {}
for ri, re in pairs:
    mu[ri] = ...
    sigma[ri] = ...
    print(f"inhibition {ri:5d} Hz: mean {mu[ri]:5.2f} mV, sd {sigma[ri]:.2f} mV, threshold 15 mV")

r = np.random.default_rng(2); dt = 0.1; n = 200000
ri, re = pairs[-1]
drive = 0.5 * r.poisson(re * dt / 1000, n) - 0.5 * r.poisson(ri * dt / 1000, n)
V = np.empty(n); v = 0.0; decay = np.exp(-dt / 20)
for k in range(n):
    v = v * decay + drive[k]; V[k] = v
print(f"simulated free voltage: mean {V[5000:].mean():.2f} mV, sd {V[5000:].std():.2f} mV")

In [ ]:
import numpy as np
def test_formulas():
    "mu and sigma follow the shot-noise formulas, with rates converted to per ms"
    ri, re = pairs[2]
    assert abs(mu[ri] - 20 * (0.5 * re - 0.5 * ri) / 1000) < 1e-9 and abs(sigma[ri] - np.sqrt(10 * (0.25 * re + 0.25 * ri) / 1000)) < 1e-9

def test_regimes():
    "with no inhibition the mean is above threshold (15 mV); with the most inhibition it is below"
    assert mu[0] > 15 and mu[10000] < 15

test_formulas()
test_regimes()
print("All checks passed")